# Sloane 237 Revelation: PDF to OSIS

Converts Nehemia Gordon's transcription and translation of the Hebrew
Revelation in the British Library (Revelation 1:1-2:13) into OSIS.

**Shelfmark.** The edition prints "MS Sloane 273", but the British Library
catalogue records the Hebrew Revelation as **Sloane MS 237** - four paper
folios in square Hebrew script, produced between 1500 and 1699, from the
bequest of Sir Hans Sloane. That is exactly the 1r-4v this text occupies, so
the headers carry 237 and cite the catalogue. The source PDF keeps its
original filename.

This is a thin driver over the tested `pdf2osis` package; the extraction lives
in `pdf2osis.glyphs`, `pdf2osis.layout` and `pdf2osis.sloane`, so the same code
is covered by `tests/test_sloane.py`.

The manuscript is fully pointed, and its PDF embeds `David` as a subset whose
ToUnicode CMap is wrong for about 12% of glyphs - sheva decodes as dagesh,
hiriq and qubuts as a shin dot. `pdf2osis.glyphs` resolves glyph IDs against
the embedded font's own `cmap` instead, which is correct and complete.

Run with the `tp` environment.

In [ ]:
from pathlib import Path

from pdf2osis import convert_pdf, get_profile

ROOT = Path.cwd().parent if Path.cwd().name == "python" else Path.cwd()
SOURCE = ROOT / "data" / "00_source_files"
OUTPUT = ROOT / "data" / "01_osis"

profile = get_profile("sloane_rev")
report = convert_pdf(profile.default_path(SOURCE), profile, OUTPUT)

print(f"{report.verses} verses across {report.chapters} chapters")
print(f"{report.note_definitions} footnotes defined")
for variant, path in report.output_paths.items():
    print(f"  {variant:20s} {path.name}")

## Source anomalies

The manuscript numbers its own verses with Hebrew letters, and those numerals do
not always agree with the printed edition's bracketed numbers. Each disagreement
is reported rather than silently resolved; the canonical numbering stays in
`osisID`/`n`, and the manuscript's own number rides on `subType`.

In [ ]:
for anomaly in report.anomalies:
    print(" -", anomaly)

## Spot check

Non-verse manuscript text is kept: the incipit as a `<title>` inside a
`<div type="introduction">`, the gate heading dividing the two chapters as a
`<title type="chapter">`, and the eight folio boundaries as
`<milestone type="pb">` at their true position in the text.

In [ ]:
from lxml import etree

from pdf2osis.osis import OSIS_NS

NS = {"osis": OSIS_NS}
tree = etree.parse(str(report.output_paths["hebrew"]))

print("incipit :", tree.xpath("//osis:div[@type='introduction']/osis:title/text()", namespaces=NS)[0])
print("gate    :", tree.xpath("//osis:title[@type='chapter']/text()", namespaces=NS)[0])
print("folios  :", tree.xpath("//osis:milestone[@type='pb']/@n", namespaces=NS))
print()
for verse in tree.xpath("//osis:verse[@sID][position() <= 3]", namespaces=NS):
    print(verse.get("osisID"), verse.tail)